### Reading data and creating features

In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel("/workspaces/hotel-demand-forecasting/data/hotel_daily_booking_data_2024_2025.xlsx",skiprows=1)
df['date'] = pd.to_datetime(df['date'])

ModuleNotFoundError: No module named 'pandas'

In [ ]:
print(df.shape)

In [ ]:
df = df.sort_values(['room_type','date'])

In [ ]:
df['month'] = df['date'].dt.month
#df['day'] = df['date'].dt.day
df['week'] = df['date'].dt.isocalendar().week.astype(int)

In [ ]:
df = df.sort_values(['room_type', 'date'])

df['lag_1'] = df.groupby('room_type')['bookings'].shift(1)


df['lag_7'] = df.groupby('room_type')['bookings'].shift(7)


df['rolling_7'] =  df.groupby('room_type')['bookings'].transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
df['price_lag_1']      = df.groupby('room_type')['price_inr'].shift(1)
df['price_lag_7']      = df.groupby('room_type')['price_inr'].shift(7)
df['price_rolling_7']  = df.groupby('room_type')['price_inr'].transform(lambda x: x.shift(1).rolling(7).mean())


df = df.dropna(subset=['lag_1', 'lag_7', 'rolling_7', 'price_lag_1', 'price_lag_7', 'price_rolling_7'])



In [ ]:
from sklearn.preprocessing import LabelEncoder
encoders = {}
for col in ['room_type','day_of_week']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

df = pd.get_dummies(df,columns=['event_name'],drop_first=True)

In [ ]:
print(encoders['day_of_week'].classes_)
print(encoders['room_type'].classes_)

In [ ]:
#future['room_type'] = encoders['room_type'].inverse_transform(future['room_type'])
#encoders['room_type'].inverse_transform(future['room_type'])


In [ ]:
print(df.columns)

#### Creating training and validation data . Validation is the last 30 days data

In [ ]:
df = df.sort_values(['room_type', 'date'])

In [ ]:
cutoff_date = (df['date'].max() - pd.Timedelta(days=30))
train_df = df[df['date'] <= cutoff_date]
val_df = df[df['date'] > cutoff_date]

print(cutoff_date)
print(train_df['date'].min(), train_df['date'].max())
print(val_df['date'].min(), val_df['date'].max())

#### XGboost fine tuning with NO PRICE features 

In [ ]:
y = df['bookings']
features = ['room_type', 'day_of_week','is_weekend','month',
            'week', 'lag_1','lag_7','rolling_7', 'event_name_Diwali',
            'event_name_New Year','event_name_Tamil New Year',
            "event_name_Valentine's Day"]

In [ ]:
X_train = train_df[features]
y_train = train_df['bookings']

X_val = val_df[features]
y_val = val_df['bookings']

In [ ]:
X_train.head()

In [ ]:
def evaluate(name, model, X, y):
    preds = model.predict(X)
    mae  = mean_absolute_error(y, preds)
    rmse = np.sqrt(mean_squared_error(y, preds))
    mape = np.mean(np.abs((y - preds) / y.clip(lower=1))) * 100
    print(f"{name:10s} | MAE: {mae:.2f} | RMSE: {rmse:.2f} | MAPE: {mape:.1f}%")
    return preds


In [ ]:
import optuna
import lightgbm as lgb
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'early_stopping_rounds': 50,   # ← moved here
        'eval_metric': 'mae',          # ← moved here too
        'random_state': 42
    }
    model = XGBRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return mean_absolute_error(y_val, model.predict(X_val))

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)
print("Best params:", study.best_params)

In [ ]:
#Best params: {'n_estimators': 624, 'max_depth': 3, 'learning_rate': 0.09033795549128364, 
#'subsample': 0.7555651059443409, 'colsample_bytree': 0.8253954417760043}

In [ ]:
best_xgb = XGBRegressor(
    **study.best_params,
    early_stopping_rounds=50,
    eval_metric='mae',
    random_state=42
)
best_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

xgb_best_pred = evaluate("XGBoost (Tuned) Test", best_xgb, X_val, y_val)
xgb_best_trainpred = evaluate("XGBoost (Tuned) Train", best_xgb, X_train, y_train)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

importance = pd.Series(best_xgb.feature_importances_, index=features)
importance.sort_values().plot(kind='barh', figsize=(8, 5))
plt.title("XGBoost Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
#print(xgb_best_pred,y_val)

In [ ]:
import matplotlib.dates as mdates
fig, axes = plt.subplots(len(val_df['room_type'].unique()), 1, 
                          figsize=(15, 4 * len(val_df['room_type'].unique())))

for ax, room in zip(axes, val_df['room_type'].unique()):
    mask = val_df['room_type'] == room
    
    ax.plot(val_df.loc[mask, 'date'], y_val[mask].values, 
            label='Actual', color='green', alpha=0.7)
    ax.plot(val_df.loc[mask, 'date'], xgb_best_pred[mask], 
            label='XGBoost', color='red', alpha=0.7)
    
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.tick_params(axis='x', rotation=45)
    ax.set_title(f"Room Type: {room}")
    ax.legend()

plt.suptitle("Actual vs Predicted Bookings for Next 30 Days by Room Type", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

### Added price features with lag and rolling metric

In [ ]:
y = df['bookings']
features_price = [
    'room_type', 'day_of_week', 'is_weekend', 'month', 'week',
    'lag_1', 'lag_7', 'rolling_7',
    'price_lag_1', 'price_lag_7', 'price_rolling_7',   # ← new
    'event_name_Diwali', 'event_name_New Year',
    'event_name_Tamil New Year', "event_name_Valentine's Day"
]


In [ ]:
X_train = train_df[features_price]
y_train = train_df['bookings']

X_val = val_df[features_price]
y_val = val_df['bookings']

#### Xgboost fine tuning with PRICE features

In [ ]:
# ── 1. Tune XGBoost ─────────────────────────────────────────────────────────
def xgb_objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 200, 1000),
        'max_depth':         trial.suggest_int('max_depth', 3, 8),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'gamma':             trial.suggest_float('gamma', 0, 0.5),
        'early_stopping_rounds': 50,
        'eval_metric': 'mae',
        'random_state': 42
    }
    model = XGBRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return mean_absolute_error(y_val, model.predict(X_val))

xgb_study = optuna.create_study(direction='minimize')
xgb_study.optimize(xgb_objective, n_trials=50)
print("Best XGBoost MAE:", xgb_study.best_value)
print("Best XGBoost params:", xgb_study.best_params)

# ── 3. Train final models with best params ──────────────────────────────────
final_xgb_price = XGBRegressor(
    **xgb_study.best_params,
    early_stopping_rounds=50,
    eval_metric='mae',
    random_state=42
)
final_xgb_price.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

In [ ]:
xgbprice_best_pred = evaluate("XGBoost (Tuned) Test", final_xgb_price, X_val, y_val)
xgbprice_best_trainpred = evaluate("XGBoost (Tuned) Train", final_xgb_price, X_train, y_train)

In [ ]:
val_result = val_df[['date','room_type','bookings']].copy()

val_result['prediction_xgb'] = xgb_best_pred
val_result['prediction_xgb_price'] = xgbprice_best_pred

val_result['median_bookings'] = val_result[['prediction_xgb','prediction_xgb_price']].median(axis=1)

In [ ]:
val_result.head(10)

In [ ]:
mae  = mean_absolute_error(val_result['bookings'], val_result['median_bookings'])
rmse = np.sqrt(mean_squared_error(val_result['bookings'], val_result['median_bookings']))
mape = np.mean(np.abs((val_result['bookings']-val_result['median_bookings']) / y.clip(lower=1))) * 100
print(f" MAE: {mae:.2f} | RMSE: {rmse:.2f} | MAPE: {mape:.1f}%")

In [ ]:
import matplotlib.pyplot as plt

room_map = {
    0: 'Deluxe',
    1: 'Standard',
    2: 'Suite'
}

for room in sorted(val_result['room_type'].unique()):

    temp = (
        val_result[val_result['room_type'] == room]
        .sort_values('date')
    )

    plt.figure(figsize=(14,6))

    plt.plot(
        temp['date'],
        temp['bookings'],
        label='Actual Bookings',
        linewidth=2
    )

    plt.plot(
        temp['date'],
        temp['prediction_xgb'],
        label='XGB'
    )

    plt.plot(
        temp['date'],
        temp['prediction_xgb_price'],
        label='XGB + Price'
    )

    plt.plot(
        temp['date'],
        temp['median_bookings'],
        label='Median Ensemble'
    )

    plt.title(f"{room_map.get(room, room)} - Actual vs Forecast")
    plt.xlabel("Date")
    plt.ylabel("Bookings")
    plt.legend()
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()

    plt.show()

In [ ]:
val_result['room_type'] = encoders['room_type'].inverse_transform(val_result['room_type'])

In [ ]:
val_result.head()

In [ ]:
#val_result.to_csv("C:/Users/Shraddha/Downloads/hotel_booking_models/hotel_booking/hotel_booking_forecast.csv")